# Exploration des Données Mammographiques
## Vision Transformer pour le Diagnostic du Cancer du Sein

Ce notebook explore le dataset et les prétraitements.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2

import config
from src.data_loader import MammogramDataLoader, create_sample_dataset
from src.preprocessing import preprocess_mammogram

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Création d'un Dataset d'Exemple

Pour tester le code, créons un petit dataset d'exemple.

In [ ]:
# Créer un dataset d'exemple
create_sample_dataset(config.RAW_DATA_DIR, num_samples_per_class=50)

print(f"Dataset créé dans: {config.RAW_DATA_DIR}")

## 2. Exploration de la Structure des Données

In [ ]:
# Compter les images par classe
class_counts = {}

for class_name in config.CLASS_NAMES:
    class_dir = Path(config.RAW_DATA_DIR) / class_name
    if class_dir.exists():
        images = list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpg'))
        class_counts[class_name] = len(images)
    else:
        class_counts[class_name] = 0

print("Distribution des classes:")
for class_name, count in class_counts.items():
    print(f"  {class_name:12s}: {count:4d} images")

total = sum(class_counts.values())
print(f"\nTotal: {total} images")

In [ ]:
# Visualiser la distribution
plt.figure(figsize=(10, 6))
plt.bar(class_counts.keys(), class_counts.values(), color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.title('Distribution des Classes', fontsize=16, fontweight='bold')
plt.xlabel('Classe', fontsize=12)
plt.ylabel('Nombre d\'Images', fontsize=12)
plt.grid(axis='y', alpha=0.3)

# Ajouter les valeurs sur les barres
for i, (name, count) in enumerate(class_counts.items()):
    plt.text(i, count + 1, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Visualisation d'Échantillons

In [ ]:
# Charger quelques échantillons de chaque classe
fig, axes = plt.subplots(3, 5, figsize=(15, 10))

for i, class_name in enumerate(config.CLASS_NAMES):
    class_dir = Path(config.RAW_DATA_DIR) / class_name
    image_files = list(class_dir.glob('*.png'))[:5]
    
    for j, img_path in enumerate(image_files):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        axes[i, j].imshow(img)
        axes[i, j].set_title(f'{class_name}', fontsize=10)
        axes[i, j].axis('off')

plt.suptitle('Échantillons par Classe', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Test du Prétraitement

In [ ]:
# Sélectionner une image
sample_image_path = list(Path(config.RAW_DATA_DIR).glob('*/*.png'))[0]
print(f"Image test: {sample_image_path}")

# Image originale
original = cv2.imread(str(sample_image_path))
original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

# Image prétraitée
preprocessed = preprocess_mammogram(
    str(sample_image_path),
    target_size=(config.IMAGE_SIZE, config.IMAGE_SIZE),
    apply_clahe_enhancement=True
)

# Visualiser
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(original)
axes[0].set_title('Image Originale', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(preprocessed)
axes[1].set_title('Image Prétraitée', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 5. Chargement avec DataLoader

In [ ]:
# Créer le data loader
data_loader = MammogramDataLoader(
    data_dir=config.RAW_DATA_DIR,
    image_size=(config.IMAGE_SIZE, config.IMAGE_SIZE),
    batch_size=config.BATCH_SIZE,
    class_names=config.CLASS_NAMES
)

# Charger et diviser les données
train_data, val_data, test_data = data_loader.load_from_directory(
    train_split=config.TRAIN_SPLIT,
    val_split=config.VAL_SPLIT,
    test_split=config.TEST_SPLIT,
    random_seed=config.RANDOM_SEED
)

In [ ]:
# Visualiser la distribution par split
splits = ['Train', 'Val', 'Test']
data_splits = [train_data[1], val_data[1], test_data[1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (split_name, labels) in enumerate(zip(splits, data_splits)):
    class_dist = [np.sum(labels == j) for j in range(len(config.CLASS_NAMES))]
    
    axes[i].bar(config.CLASS_NAMES, class_dist, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    axes[i].set_title(f'{split_name} Set', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Nombre d\'Images')
    axes[i].grid(axis='y', alpha=0.3)
    
    for j, count in enumerate(class_dist):
        axes[i].text(j, count + 0.5, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Créer un TensorFlow Dataset

In [ ]:
# Créer le dataset TensorFlow
train_dataset = data_loader.create_tf_dataset(
    train_data[0], train_data[1],
    shuffle=True, augment=True
)

# Visualiser un batch
for images, labels in train_dataset.take(1):
    print(f"Batch shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    
    # Afficher les premières images
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i in range(8):
        axes[i].imshow(images[i])
        label_idx = np.argmax(labels[i])
        axes[i].set_title(f'{config.CLASS_NAMES[label_idx]}', fontsize=12)
        axes[i].axis('off')
    
    plt.suptitle('Batch d\'Entraînement (avec Augmentation)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 7. Statistiques des Images

In [ ]:
# Analyser quelques images
sample_paths = train_data[0][:20]
pixel_means = []
pixel_stds = []

for path in sample_paths:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        pixel_means.append(img.mean())
        pixel_stds.append(img.std())

print(f"Moyenne des pixels: {np.mean(pixel_means):.2f} ± {np.std(pixel_means):.2f}")
print(f"Écart-type des pixels: {np.mean(pixel_stds):.2f} ± {np.std(pixel_stds):.2f}")

# Histogrammes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(pixel_means, bins=20, color='skyblue', edgecolor='black')
axes[0].set_title('Distribution des Moyennes', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Moyenne des Pixels')
axes[0].set_ylabel('Fréquence')

axes[1].hist(pixel_stds, bins=20, color='lightcoral', edgecolor='black')
axes[1].set_title('Distribution des Écarts-types', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Écart-type des Pixels')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

## Conclusion

Ce notebook a exploré:
- ✓ La structure et la distribution du dataset
- ✓ Le prétraitement des images mammographiques
- ✓ La création de datasets TensorFlow
- ✓ L'augmentation de données

Prochaine étape: **02_training.ipynb** pour entraîner le modèle Vision Transformer.